## Optimized Stacking Ensemble Workflow

This notebook upgrades the training flow to support:
1. Group-aware or time-aware validation.
2. Optuna tuning for the two-stage XGBoost classifier and regressor.
3. Threshold tuning with a precision-recall tradeoff.
4. Exporting one deployment-ready artifact to the model/stacking_ensemble folder.

The saved package contains the tuned stacking model, the tuned two-stage classifier and regressor, the tuned threshold, and the blend weight needed by the web app.

In [20]:
import re
import time
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
try:
    import optuna
except ModuleNotFoundError:
    import sys
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])
    import optuna
import pandas as pd
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [21]:
DATA_PATH = Path("../data/bleaching_model_ready.csv")
ARTIFACT_DIR = Path("../model/stacking_ensemble")
ARTIFACT_PATH = ARTIFACT_DIR / "Stacking_Ensemble_model.joblib"

SPLIT_MODE = "group"
N_SPLITS = 4 
RANDOM_STATE = 42
SPATIAL_GRID_DEGREES = 1.0
CLASSIFIER_TUNING_METRIC = "roc_auc"
CLASSIFIER_OPTUNA_TRIALS = 20
REGRESSOR_OPTUNA_TRIALS = 20
THRESHOLD_BETA = 2.0
MIN_THRESHOLD_PRECISION = 0.60
THRESHOLD_GRID_MIN = 0.10
THRESHOLD_GRID_MAX = 0.90
THRESHOLD_GRID_STEPS = 81
BLEND_GRID = np.linspace(0.0, 1.0, 21)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH: {DATA_PATH}")
print(f"ARTIFACT_DIR: {ARTIFACT_DIR}")
print(f"ARTIFACT_PATH: {ARTIFACT_PATH}")
print(f"SPLIT_MODE: {SPLIT_MODE}")
print(f"N_SPLITS: {N_SPLITS}")

DATA_PATH: ..\data\bleaching_model_ready.csv
ARTIFACT_DIR: ..\model\stacking_ensemble
ARTIFACT_PATH: ..\model\stacking_ensemble\Stacking_Ensemble_model.joblib
SPLIT_MODE: group
N_SPLITS: 4


In [22]:
def sanitize_feature_name(name):
    cleaned = re.sub(r"[^0-9a-zA-Z_]+", "_", name.strip())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned or "feature"

print("1. Loading and preparing data...")
df = pd.read_csv(DATA_PATH).copy()
df["Bleaching_Flag"] = (df["Percent_Bleaching"] > 0).astype(int)
lat_bin = (np.floor(df["Latitude_Degrees"] / SPATIAL_GRID_DEGREES) * SPATIAL_GRID_DEGREES).round(1)
lon_bin = (np.floor(df["Longitude_Degrees"] / SPATIAL_GRID_DEGREES) * SPATIAL_GRID_DEGREES).round(1)
df["Spatial_Group"] = lat_bin.astype(str) + "_" + lon_bin.astype(str)
df["Date_Sort"] = pd.to_datetime(
    dict(
        year=df["Date_Year"].astype(int),
        month=df["Date_Month"].astype(int),
        day=df["Date_Day"].astype(int),
    ),
    errors="coerce",
)
df = df.sort_values(["Date_Sort", "Date_Year", "Date_Month", "Date_Day"]).reset_index(drop=True)
original_feature_columns = [col for col in df.columns if col not in {"Percent_Bleaching", "Bleaching_Flag", "Spatial_Group", "Date_Sort"}]
sanitized_feature_columns = [sanitize_feature_name(col) for col in original_feature_columns]
X = df[original_feature_columns].copy()
X.columns = sanitized_feature_columns
bool_cols = X.select_dtypes(include="bool").columns
if len(bool_cols) > 0:
    X[bool_cols] = X[bool_cols].astype(int)
y = df["Percent_Bleaching"].copy()
y_log = np.log1p(y)
y_binary = df["Bleaching_Flag"].copy()
groups = df["Spatial_Group"].copy()
print(f"   -> rows: {len(df):,}")
print(f"   -> features: {len(sanitized_feature_columns)}")
print(f"   -> zero-bleaching share: {y.eq(0).mean() * 100:.2f}%")
print(f"   -> unique spatial groups: {groups.nunique():,}")
print(f"   -> years: {int(df['Date_Year'].min())} to {int(df['Date_Year'].max())}")

1. Loading and preparing data...
   -> rows: 34,515
   -> features: 61
   -> zero-bleaching share: 48.18%
   -> unique spatial groups: 649
   -> years: 1998 to 2019


In [23]:
def make_outer_splitter(split_mode):
    if split_mode == "group":
        return GroupKFold(n_splits=N_SPLITS)
    if split_mode == "time":
        return TimeSeriesSplit(n_splits=N_SPLITS)
    raise ValueError("SPLIT_MODE must be 'group' or 'time'")

def iter_splits(splitter, X_data, y_data, groups_data=None):
    if groups_data is None:
        return splitter.split(X_data, y_data)
    return splitter.split(X_data, y_data, groups_data)

def regression_metrics(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }

def classification_metrics(y_true, y_pred, y_proba):
    metrics = {
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Average_Precision": float(average_precision_score(y_true, y_proba)),
    }
    metrics["ROC_AUC"] = float(roc_auc_score(y_true, y_proba)) if pd.Series(y_true).nunique() > 1 else np.nan
    return metrics

def tune_split(X_data, y_reg, y_cls, split_mode, groups_data=None):
    splitter = make_outer_splitter(split_mode)
    train_idx, valid_idx = next(iter(iter_splits(splitter, X_data, y_reg, groups_data)))
    return {
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "X_train": X_data.iloc[train_idx].copy(),
        "X_valid": X_data.iloc[valid_idx].copy(),
        "y_train": y_reg.iloc[train_idx].copy(),
        "y_valid": y_reg.iloc[valid_idx].copy(),
        "y_train_log": np.log1p(y_reg.iloc[train_idx].copy()),
        "y_valid_log": np.log1p(y_reg.iloc[valid_idx].copy()),
        "y_train_flag": y_cls.iloc[train_idx].copy(),
        "y_valid_flag": y_cls.iloc[valid_idx].copy(),
    }

def build_rf_regressor():
    return RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_split=10,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

def build_lgbm_regressor():
    return LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )

def build_xgb_classifier(params=None):
    base_params = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 6,
        "min_child_weight": 1.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
    }
    if params:
        base_params.update(params)
    return XGBClassifier(**base_params)

def build_xgb_regressor(params=None):
    base_params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 8,
        "min_child_weight": 1.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
    }
    if params:
        base_params.update(params)
    return XGBRegressor(**base_params)

def build_stacking_regressor(split_mode, xgb_regressor_params=None):
    inner_cv = TimeSeriesSplit(n_splits=N_SPLITS) if split_mode == "time" else KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    return StackingRegressor(
        estimators=[
            ("rf", build_rf_regressor()),
            ("xgb", build_xgb_regressor(xgb_regressor_params)),
            ("lgbm", build_lgbm_regressor()),
        ],
        final_estimator=Pipeline(steps=[("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0))]),
        cv=inner_cv,
        n_jobs=-1,
        passthrough=False,
    )

def predict_regression_model(model, X_data):
    preds = np.expm1(model.predict(X_data))
    return np.clip(preds, 0, 100)

def predict_two_stage(classifier, regressor, X_data, threshold):
    risk_proba = classifier.predict_proba(X_data)[:, 1]
    risk_flag = (risk_proba >= threshold).astype(int)
    severity = np.zeros(len(X_data), dtype=float)
    if risk_flag.any():
        severity[risk_flag == 1] = np.expm1(regressor.predict(X_data.loc[risk_flag == 1]))
    severity = np.clip(severity, 0, 100)
    return risk_proba, risk_flag, severity

def blended_prediction(stacking_pred, two_stage_pred, alpha):
    return np.clip(alpha * stacking_pred + (1.0 - alpha) * two_stage_pred, 0, 100)

def choose_blend_alpha(y_true, stacking_pred, two_stage_pred):
    rows = []
    for alpha in BLEND_GRID:
        blended = blended_prediction(stacking_pred, two_stage_pred, alpha)
        metrics = regression_metrics(y_true, blended)
        rows.append({"alpha": float(alpha), **metrics})
    blend_df = pd.DataFrame(rows).sort_values(["RMSE", "MAE", "R2"], ascending=[True, True, False]).reset_index(drop=True)
    return float(blend_df.loc[0, "alpha"]), blend_df

def tune_threshold(y_true, y_proba, beta=2.0, min_precision=0.60):
    threshold_grid = np.linspace(THRESHOLD_GRID_MIN, THRESHOLD_GRID_MAX, THRESHOLD_GRID_STEPS)
    rows = []
    for threshold in threshold_grid:
        preds = (y_proba >= threshold).astype(int)
        precision = precision_score(y_true, preds, zero_division=0)
        recall = recall_score(y_true, preds, zero_division=0)
        if precision == 0 and recall == 0:
            fbeta = 0.0
        else:
            beta_sq = beta ** 2
            fbeta = (1 + beta_sq) * precision * recall / max(beta_sq * precision + recall, 1e-9)
        rows.append({"threshold": float(threshold), "precision": float(precision), "recall": float(recall), "f_beta": float(fbeta)})
    threshold_df = pd.DataFrame(rows)
    eligible = threshold_df[threshold_df["precision"] >= min_precision]
    candidate_df = eligible if not eligible.empty else threshold_df
    candidate_df = candidate_df.sort_values(["f_beta", "recall", "precision"], ascending=[False, False, False]).reset_index(drop=True)
    return float(candidate_df.loc[0, "threshold"]), threshold_df

def evaluate_regressor_cv(model_name, model_factory, X_data, y_data, splitter, groups_data=None):
    rows = []
    for fold, (train_idx, test_idx) in enumerate(iter_splits(splitter, X_data, y_data, groups_data), start=1):
        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]
        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]
        model = model_factory()
        start_time = time.time()
        model.fit(X_train, np.log1p(y_train))
        fit_time = time.time() - start_time
        y_pred = predict_regression_model(model, X_test)
        metrics = regression_metrics(y_test, y_pred)
        rows.append({"Model": model_name, "Fold": fold, "Fit_Time_Sec": fit_time, **metrics})
    return pd.DataFrame(rows)

def evaluate_two_stage_cv(model_name, classifier_params, regressor_params, threshold, X_data, y_data, y_flag, splitter, groups_data=None):
    rows = []
    for fold, (train_idx, test_idx) in enumerate(iter_splits(splitter, X_data, y_data, groups_data), start=1):
        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]
        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]
        y_train_flag = y_flag.iloc[train_idx]
        y_test_flag = y_flag.iloc[test_idx]
        classifier = build_xgb_classifier(classifier_params)
        regressor = build_xgb_regressor(regressor_params)
        start_time = time.time()
        classifier.fit(X_train, y_train_flag)
        positive_train_mask = y_train_flag == 1
        if int(positive_train_mask.sum()) > 0:
            regressor.fit(X_train.loc[positive_train_mask], np.log1p(y_train.loc[positive_train_mask]))
        fit_time = time.time() - start_time
        risk_proba, risk_flag, severity_pred = predict_two_stage(classifier, regressor, X_test, threshold)
        reg_metrics = regression_metrics(y_test, severity_pred)
        cls_metrics = classification_metrics(y_test_flag, risk_flag, risk_proba)
        rows.append({"Model": model_name, "Fold": fold, "Fit_Time_Sec": fit_time, **reg_metrics, **cls_metrics, "Predicted_Positive_Rate": float(risk_flag.mean()), "Actual_Positive_Rate": float(y_test_flag.mean())})
    return pd.DataFrame(rows)

def evaluate_hybrid_cv(model_name, xgb_regressor_params, classifier_params, regressor_params, threshold, blend_alpha, X_data, y_data, y_flag, splitter, groups_data=None):
    rows = []
    for fold, (train_idx, test_idx) in enumerate(iter_splits(splitter, X_data, y_data, groups_data), start=1):
        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]
        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]
        y_train_flag = y_flag.iloc[train_idx]
        stacking_model = build_stacking_regressor(SPLIT_MODE, xgb_regressor_params)
        classifier = build_xgb_classifier(classifier_params)
        regressor = build_xgb_regressor(regressor_params)
        start_time = time.time()
        stacking_model.fit(X_train, np.log1p(y_train))
        classifier.fit(X_train, y_train_flag)
        positive_train_mask = y_train_flag == 1
        if int(positive_train_mask.sum()) > 0:
            regressor.fit(X_train.loc[positive_train_mask], np.log1p(y_train.loc[positive_train_mask]))
        fit_time = time.time() - start_time
        stacking_pred = predict_regression_model(stacking_model, X_test)
        _, _, two_stage_pred = predict_two_stage(classifier, regressor, X_test, threshold)
        hybrid_pred = blended_prediction(stacking_pred, two_stage_pred, blend_alpha)
        metrics = regression_metrics(y_test, hybrid_pred)
        rows.append({"Model": model_name, "Fold": fold, "Fit_Time_Sec": fit_time, **metrics})
    return pd.DataFrame(rows)

def summarize_regression_results(result_df):
    return result_df.groupby("Model", as_index=False).agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        Fit_Time_mean=("Fit_Time_Sec", "mean"),
    ).sort_values(["RMSE_mean", "MAE_mean", "R2_mean"], ascending=[True, True, False]).reset_index(drop=True)

def summarize_two_stage_results(result_df):
    return result_df.groupby("Model", as_index=False).agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        Precision_mean=("Precision", "mean"),
        Recall_mean=("Recall", "mean"),
        F1_mean=("F1", "mean"),
        ROC_AUC_mean=("ROC_AUC", "mean"),
        Average_Precision_mean=("Average_Precision", "mean"),
        Fit_Time_mean=("Fit_Time_Sec", "mean"),
    ).reset_index(drop=True)

In [24]:
print("2. Building the tuning split...")
split_groups = groups if SPLIT_MODE == "group" else None
split_bundle = tune_split(X, y, y_binary, SPLIT_MODE, split_groups)
X_train_tune = split_bundle["X_train"]
X_valid_tune = split_bundle["X_valid"]
y_train_tune = split_bundle["y_train"]
y_valid_tune = split_bundle["y_valid"]
y_train_tune_log = split_bundle["y_train_log"]
y_valid_tune_log = split_bundle["y_valid_log"]
y_train_tune_flag = split_bundle["y_train_flag"]
y_valid_tune_flag = split_bundle["y_valid_flag"]
print(f"   -> tuning train rows: {len(X_train_tune):,}")
print(f"   -> tuning valid rows: {len(X_valid_tune):,}")
print(f"   -> tuning train positives: {int(y_train_tune_flag.sum()):,}")
print(f"   -> tuning valid positives: {int(y_valid_tune_flag.sum()):,}")

2. Building the tuning split...
   -> tuning train rows: 25,886
   -> tuning valid rows: 8,629
   -> tuning train positives: 13,279
   -> tuning valid positives: 4,607


In [25]:
print("3. Tuning the stage-1 classifier with Optuna...")
positive_ratio = float(y_train_tune_flag.mean())
negative_positive_ratio = float((1.0 - positive_ratio) / max(positive_ratio, 1e-6))
def classifier_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", max(0.5, negative_positive_ratio * 0.5), max(1.0, negative_positive_ratio * 1.5)),
    }
    model = build_xgb_classifier(params)
    model.fit(X_train_tune, y_train_tune_flag)
    valid_proba = model.predict_proba(X_valid_tune)[:, 1]
    if CLASSIFIER_TUNING_METRIC == "f1":
        valid_pred = (valid_proba >= 0.5).astype(int)
        return f1_score(y_valid_tune_flag, valid_pred, zero_division=0)
    return roc_auc_score(y_valid_tune_flag, valid_proba)
classifier_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
classifier_study.optimize(classifier_objective, n_trials=CLASSIFIER_OPTUNA_TRIALS, show_progress_bar=False)
best_classifier_params = classifier_study.best_params
print(f"   -> best classifier score ({CLASSIFIER_TUNING_METRIC}): {classifier_study.best_value:.5f}")
display(pd.DataFrame([best_classifier_params]))

print("4. Tuning the classification threshold...")
tuned_classifier = build_xgb_classifier(best_classifier_params)
tuned_classifier.fit(X_train_tune, y_train_tune_flag)
valid_risk_proba = tuned_classifier.predict_proba(X_valid_tune)[:, 1]
tuned_threshold, threshold_df = tune_threshold(y_true=y_valid_tune_flag, y_proba=valid_risk_proba, beta=THRESHOLD_BETA, min_precision=MIN_THRESHOLD_PRECISION)
valid_risk_pred = (valid_risk_proba >= tuned_threshold).astype(int)
threshold_metrics = classification_metrics(y_valid_tune_flag, valid_risk_pred, valid_risk_proba)
print(f"   -> tuned threshold: {tuned_threshold:.4f}")
print(f"   -> validation precision: {threshold_metrics['Precision']:.4f}")
print(f"   -> validation recall: {threshold_metrics['Recall']:.4f}")
print(f"   -> validation F1: {threshold_metrics['F1']:.4f}")
display(threshold_df.sort_values(["f_beta", "recall"], ascending=[False, False]).head(10).round(4))

print("5. Tuning the stage-2 regressor with Optuna...")
positive_train_mask = y_train_tune > 0
positive_valid_mask = y_valid_tune > 0
X_train_positive = X_train_tune.loc[positive_train_mask]
y_train_positive = y_train_tune.loc[positive_train_mask]
X_valid_positive = X_valid_tune.loc[positive_valid_mask]
y_valid_positive = y_valid_tune.loc[positive_valid_mask]
def regressor_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 150, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    model = build_xgb_regressor(params)
    model.fit(X_train_positive, np.log1p(y_train_positive))
    valid_pred = predict_regression_model(model, X_valid_positive)
    rmse = np.sqrt(mean_squared_error(y_valid_positive, valid_pred))
    return float(rmse)
regressor_study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
regressor_study.optimize(regressor_objective, n_trials=REGRESSOR_OPTUNA_TRIALS, show_progress_bar=False)
best_regressor_params = regressor_study.best_params
print(f"   -> best regressor RMSE: {regressor_study.best_value:.5f}")
display(pd.DataFrame([best_regressor_params]))

3. Tuning the stage-1 classifier with Optuna...
   -> best classifier score (roc_auc): 0.84854


,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,scale_pos_weight
0,434,0.030544,7,2.701144,0.760382,0.626006,3.319048,0.002901,1.70712,1.312843


4. Tuning the classification threshold...
   -> tuned threshold: 0.2400
   -> validation precision: 0.6026
   -> validation recall: 0.9570
   -> validation F1: 0.7395


,threshold,precision,recall,f_beta
10,0.20,0.5814,0.9757,0.8591
8,0.18,0.5723,0.9818,0.8589
9,0.19,0.5765,0.9785,0.8587
11,0.21,0.5865,0.9707,0.8583
7,0.17,0.5687,0.9833,0.8582
12,0.22,0.5908,0.9666,0.8575
13,0.23,0.5978,0.9614,0.8571
6,0.16,0.5636,0.9842,0.8564
5,0.15,0.5596,0.9872,0.8563
14,0.24,0.6026,0.9570,0.8563


5. Tuning the stage-2 regressor with Optuna...
   -> best regressor RMSE: 20.29322


,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,439,0.08138,6,8.423438,0.868798,0.94925,0.559611,0.000034,0.485451


In [26]:
print("6. Comparing tuned models on the tuning validation split...")
tuned_regressor = build_xgb_regressor(best_regressor_params)
tuned_regressor.fit(X_train_positive, np.log1p(y_train_positive))
tuned_stacking = build_stacking_regressor(SPLIT_MODE, best_regressor_params)
tuned_stacking.fit(X_train_tune, y_train_tune_log)
stacking_valid_pred = predict_regression_model(tuned_stacking, X_valid_tune)
valid_risk_proba, valid_risk_flag, two_stage_valid_pred = predict_two_stage(tuned_classifier, tuned_regressor, X_valid_tune, tuned_threshold)
blend_alpha, blend_df = choose_blend_alpha(y_valid_tune, stacking_valid_pred, two_stage_valid_pred)
hybrid_valid_pred = blended_prediction(stacking_valid_pred, two_stage_valid_pred, blend_alpha)
validation_rows = [
    {"Model": "Stacking_TunedXGB", **regression_metrics(y_valid_tune, stacking_valid_pred)},
    {"Model": "TwoStage_TunedXGB", **regression_metrics(y_valid_tune, two_stage_valid_pred)},
    {"Model": "Hybrid_Stacking_TwoStage", **regression_metrics(y_valid_tune, hybrid_valid_pred)},
]
validation_summary_df = pd.DataFrame(validation_rows).sort_values(["RMSE", "MAE", "R2"], ascending=[True, True, False]).reset_index(drop=True)
print(f"   -> tuned blend alpha: {blend_alpha:.2f}")
display(validation_summary_df.round(4))
display(blend_df.round(4))

print("7. Benchmarking all models with the tuned parameters...")
outer_splitter = make_outer_splitter(SPLIT_MODE)
outer_groups = groups if SPLIT_MODE == "group" else None
regression_results = []
regression_results.append(evaluate_regressor_cv("RandomForest", build_rf_regressor, X, y, outer_splitter, outer_groups))
regression_results.append(evaluate_regressor_cv("LightGBM", build_lgbm_regressor, X, y, outer_splitter, outer_groups))
regression_results.append(evaluate_regressor_cv("XGBoost_Tuned", lambda: build_xgb_regressor(best_regressor_params), X, y, outer_splitter, outer_groups))
regression_results.append(evaluate_regressor_cv("Stacking_TunedXGB", lambda: build_stacking_regressor(SPLIT_MODE, best_regressor_params), X, y, outer_splitter, outer_groups))
regression_results.append(evaluate_hybrid_cv("Hybrid_Stacking_TwoStage", best_regressor_params, best_classifier_params, best_regressor_params, tuned_threshold, blend_alpha, X, y, y_binary, outer_splitter, outer_groups))
regression_folds_df = pd.concat(regression_results, ignore_index=True)
regression_summary_df = summarize_regression_results(regression_folds_df)
two_stage_folds_df = evaluate_two_stage_cv("TwoStage_TunedXGB", best_classifier_params, best_regressor_params, tuned_threshold, X, y, y_binary, outer_splitter, outer_groups)
two_stage_summary_df = summarize_two_stage_results(two_stage_folds_df)
combined_summary_df = pd.concat([regression_summary_df.assign(Family="Regression"), two_stage_summary_df.assign(Family="TwoStage")], ignore_index=True, sort=False).sort_values(["RMSE_mean", "MAE_mean", "R2_mean"], ascending=[True, True, False]).reset_index(drop=True)
display(regression_summary_df.round(4))
display(two_stage_summary_df.round(4))
display(combined_summary_df.round(4))

6. Comparing tuned models on the tuning validation split...
   -> tuned blend alpha: 0.30


,Model,RMSE,MAE,R2
0,Hybrid_Stacking_TwoStage,14.9827,7.3843,0.4595
1,TwoStage_TunedXGB,15.1073,7.6519,0.4504
2,Stacking_TunedXGB,15.5942,7.1997,0.4144


,alpha,RMSE,MAE,R2
0,0.30,14.9827,7.3843,0.4595
1,0.35,14.9848,7.3493,0.4593
2,0.25,14.9872,7.4220,0.4591
3,0.40,14.9934,7.3168,0.4587
4,0.20,14.9982,7.4624,0.4583
5,0.45,15.0085,7.2871,0.4576
6,0.15,15.0158,7.5055,0.4571
7,0.50,15.0301,7.2604,0.4560
8,0.10,15.0398,7.5516,0.4553
9,0.55,15.0582,7.2388,0.4540


7. Benchmarking all models with the tuned parameters...


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std,Fit_Time_mean
0,Hybrid_Stacking_TwoStage,15.4069,1.2425,7.6579,0.8849,0.4144,0.0352,120.9650
1,Stacking_TunedXGB,15.7032,1.0993,7.3104,0.8630,0.3916,0.0161,114.2275
2,XGBoost_Tuned,15.7814,1.2514,7.2177,0.8547,0.3859,0.0251,1.3201
3,LightGBM,15.9735,1.3487,7.2337,0.8757,0.3711,0.0325,0.8183
4,RandomForest,16.1826,1.0851,7.4986,0.8954,0.3538,0.0104,13.4804


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std,Precision_mean,Recall_mean,F1_mean,ROC_AUC_mean,Average_Precision_mean,Fit_Time_mean
0,TwoStage_TunedXGB,15.5918,1.2714,7.9596,0.8876,0.4002,0.0385,0.5826,0.937,0.7185,0.8349,0.8775,2.302


,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std,Fit_Time_mean,Family,Precision_mean,Recall_mean,F1_mean,ROC_AUC_mean,Average_Precision_mean
0,Hybrid_Stacking_TwoStage,15.4069,1.2425,7.6579,0.8849,0.4144,0.0352,120.9650,Regression,NaN,NaN,NaN,NaN,NaN
1,TwoStage_TunedXGB,15.5918,1.2714,7.9596,0.8876,0.4002,0.0385,2.3020,TwoStage,0.5826,0.937,0.7185,0.8349,0.8775
2,Stacking_TunedXGB,15.7032,1.0993,7.3104,0.8630,0.3916,0.0161,114.2275,Regression,NaN,NaN,NaN,NaN,NaN
3,XGBoost_Tuned,15.7814,1.2514,7.2177,0.8547,0.3859,0.0251,1.3201,Regression,NaN,NaN,NaN,NaN,NaN
4,LightGBM,15.9735,1.3487,7.2337,0.8757,0.3711,0.0325,0.8183,Regression,NaN,NaN,NaN,NaN,NaN
5,RandomForest,16.1826,1.0851,7.4986,0.8954,0.3538,0.0104,13.4804,Regression,NaN,NaN,NaN,NaN,NaN


In [27]:
print("8. Training final deployment models on the full dataset...")
final_stacking_model = build_stacking_regressor(SPLIT_MODE, best_regressor_params)
final_classifier_model = build_xgb_classifier(best_classifier_params)
final_regressor_model = build_xgb_regressor(best_regressor_params)
final_stacking_model.fit(X, y_log)
final_classifier_model.fit(X, y_binary)
positive_full_mask = y > 0
final_regressor_model.fit(X.loc[positive_full_mask], y_log.loc[positive_full_mask])
final_artifact = {
    "artifact_name": "Stacking_Ensemble_model",
    "artifact_version": 1,
    "created_at": datetime.utcnow().isoformat() + "Z",
    "split_mode": SPLIT_MODE,
    "feature_names": sanitized_feature_columns,
    "original_feature_names": original_feature_columns,
    "target_name": "Percent_Bleaching",
    "target_transform": "log1p",
    "prediction_clip": [0, 100],
    "two_stage_threshold": float(tuned_threshold),
    "blend_alpha": float(blend_alpha),
    "classifier_best_params": best_classifier_params,
    "regressor_best_params": best_regressor_params,
    "stacking_model": final_stacking_model,
    "two_stage_classifier": final_classifier_model,
    "two_stage_regressor": final_regressor_model,
    "deployment_mode": "hybrid_stacking_two_stage",
    "prediction_recipe": {
        "stacking_prediction": "expm1(stacking_model.predict(X)) clipped to [0, 100]",
        "two_stage_probability": "two_stage_classifier.predict_proba(X)[:, 1]",
        "two_stage_flag": "probability >= two_stage_threshold",
        "two_stage_severity": "expm1(two_stage_regressor.predict(X_positive)) clipped to [0, 100]",
        "final_prediction": "blend_alpha * stacking_prediction + (1 - blend_alpha) * two_stage_severity",
    },
}
joblib.dump(final_artifact, ARTIFACT_PATH)
print(f"   -> saved deploy artifact: {ARTIFACT_PATH}")
print("9. Done. This notebook now exports only the web deployment artifact.")

8. Training final deployment models on the full dataset...
   -> saved deploy artifact: ..\model\stacking_ensemble\Stacking_Ensemble_model.joblib
9. Done. This notebook now exports only the web deployment artifact.
